# Week 5 · Fit

A model is a shape and a rule for being wrong. This lab builds four of them on
the hotel data, and reads every number out loud.

Six steps:

1. the simplest shape there is, fitted twice
2. a model you can read to a hotel manager
3. what one strange booking does to two different losses
4. the same idea for yes or no, in odds
5. one coefficient asked to describe two worlds
6. five sentences you could defend

**Before you start:** `python data/download.py --hotels` from the repository root.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hotels

pd.set_option("display.width", 120)

df = hotels.block_2()
rate = hotels.daily_rate(df)
cancel = hotels.cancellation(df)

rtrain, rvalid, _ = hotels.splits(rate)
ctrain, cvalid, _ = hotels.splits(cancel)
print(f"rate   train {len(rtrain):,}  valid {len(rvalid):,}")
print(f"cancel train {len(ctrain):,}  valid {len(cvalid):,}")

## 1 · The simplest shape there is

$$\text{rate} = w_0 + w_1 \times \text{guests}$$

Two numbers. Fit them the easy way first.

In [ ]:
# TODO: fit LinearRegression on rtrain[["total_guests"]] -> rtrain["target"].
# Print the intercept and the coefficient, then R2 and MAE on rvalid, and the
# MAE you get by predicting the training mean for everyone.
simple = ...
w0, w1 = ...

Now the hard way, so the formula stops being magic. Write the loss yourself and
search a grid of candidate models for the lowest one.

In [ ]:
# TODO: write squared_loss(w0, w1, x, y) -- the mean of (prediction - truth)^2.
# Then build a grid around the fitted values, score every pair, and find the
# lowest. It should land on the same answer as LinearRegression.
def squared_loss(w0, w1, x, y):
    ...

In [ ]:
assert abs(grid0[j] - w0) < 1.0 and abs(grid1[i] - w1) < 0.5, \
    "the grid minimum should be within a euro of the formula's answer"
print("check 1 passed: searching and solving agree")

## 2 · A model you can read

Five numeric columns, four categorical ones. The point is not the score, it is
that every number in it means something in euros.

In [ ]:
# TODO: build a pipeline -- median impute the numeric columns, one-hot the
# categorical ones with drop="first", then LinearRegression. Fit on rtrain.
# Pull the coefficients out into a Series indexed by feature name, and report
# R2 and MAE on rvalid.
NUM = ["lead_time", "total_nights", "adults", "children", "total_of_special_requests"]
CAT = ["hotel", "arrival_month", "market_segment", "reserved_room_type"]

readable = ...
coefs = ...

In [ ]:
assert r2_score(rvalid["target"], pred) > 0.38, "expected R2 above 0.38 on validation"
print("check 2 passed: the readable model clears the floor comfortably")

**Write down five of these in plain English.** One sentence each, in euros, in
the form *an extra X adds/removes €Y a night, holding everything else still*.

Two to look at before you do: `children` against `adults`, and the sign on
`total_nights`.

In [ ]:
# your five sentences, as comments or a markdown cell:
#
# 1.
# 2.
# 3.
# 4.
# 5.

## 3 · One strange booking, two losses

The file contains exactly one booking at €5,400 a night. `daily_rate()` leaves
it out. Put it back and watch what each loss does.

In [ ]:
# TODO: take 4,000 training bookings. Fit two models of adr on total_guests --
# LinearRegression (squared error) and QuantileRegressor(quantile=0.5, alpha=0,
# solver="highs") (absolute error). Then add one row at EUR 5,400 for a
# 2-guest booking and fit both again. Put the four intercepts and four slopes
# in a DataFrame called `comparison`.

In [ ]:
moved_sq = abs(comparison.loc["with the outlier", "squared intercept"]
               - comparison.loc["clean", "squared intercept"])
moved_abs = abs(comparison.loc["with the outlier", "absolute intercept"]
                - comparison.loc["clean", "absolute intercept"])
print(f"squared intercept moved EUR {moved_sq:.2f}")
print(f"absolute intercept moved EUR {moved_abs:.2f}")
assert moved_abs < 0.01 < moved_sq, "expected squared error to move and absolute error not to"
print("check 3 passed: one row in four thousand moved one fit and not the other")

**Predict before you run the next cell.** You are about to fit the same two
models on the *full* training set, where the €5,400 booking is one row in
76,593 instead of one in 4,000. Does the gap between the two losses get bigger
or smaller, and why?

In [ ]:
# TODO: your prediction, in one line, then run it and see.

## 4 · The same idea, for yes or no

Cancellation is not euros. Same straight line underneath, squashed into a
probability, and the coefficients come out as multipliers on **odds**.

In [ ]:
# TODO: same shape as step 2, but scale the numeric columns, use
# LogisticRegression, and report PR AUC, lift and ROC AUC on cvalid.
# Then turn the coefficients into odds multipliers with np.exp and show the
# eight furthest from 1.
CNUM = ["lead_time", "total_nights", "previous_cancellations",
        "total_of_special_requests", "is_repeated_guest"]
CCAT = ["hotel", "deposit_type", "market_segment", "customer_type"]

logistic = ...
odds = ...

In [ ]:
assert ap / cvalid["target"].mean() > 1.8, "expected a lift above 1.8"
print("check 4 passed: the cancellation model is about twice as good as guessing")

Now the conditional part. Fit `deposit_type` **alone**, with nothing else in
the model, and compare its odds multiplier to the one above.

In [ ]:
# TODO: fit a model on deposit_type alone and compare its Non Refund odds
# multiplier with the one from the full model. Then look at mean lead_time by
# deposit_type and explain the difference in one sentence.

## 5 · One coefficient, two worlds

The model has a single August coefficient and two very different hotels. Find
out what it is hiding.

In [ ]:
# TODO: mean adr by arrival_month and hotel, as a pivot table, plus a column
# for the difference between them. Call it `season`.

In [ ]:
ratio_resort = season.loc[8, "Resort Hotel"] / season.loc[11, "Resort Hotel"]
ratio_city = season.loc[8, "City Hotel"] / season.loc[11, "City Hotel"]
print(f"resort: August is {ratio_resort:.1f}x November")
print(f"city:   August is {ratio_city:.1f}x November")
assert ratio_resort > 3 and ratio_city < 1.5, "expected the resort to swing and the city not to"
print("check 5 passed: one month coefficient cannot be right for both")

Now fit the two hotels separately and compare each one's August coefficient
with the pooled one.

In [ ]:
# TODO: fit the same readable model separately on each hotel (drop `hotel`
# from the categorical columns -- it is constant inside each fit) and put the
# three August coefficients side by side: pooled, city, resort.

## 6 · Five sentences you could defend

Write five things your models let you say to a hotel manager, and one thing
they do not. Be specific: numbers, units, and the clause *holding everything
else still* where it belongs.

The one they do not support is the important one. Candidates: anything causal,
anything about a hotel not in this data, anything about August at the resort
specifically, and anything about what would happen if the hotel changed a
policy.

In [ ]:
# Five you can defend:
#
# 1.
# 2.
# 3.
# 4.
# 5.
#
# One you cannot, and why:
#

---

**Next week:** the formula goes away. When you cannot jump to the bottom of the
bowl you have to walk, and walking fails in three visible ways.